# 13a — OOD-Controlled Blend Test

This notebook tests the **first extrapolation-control idea** only:

1. Detect OOD test scenarios using nearest-neighbor distance to the training set.
2. Compare three forward submission candidates:
   - `prediction_submission.csv` — conservative ExtraTrees final model.
   - `prediction_submission_hybrid_mlp_distance_w_0.65.csv` — aggressive MLP-distance hybrid.
   - a controlled blend between the two.
3. Create several blend candidates and inspect their test-set distributions.
4. Optionally create a new candidate submission file.

The goal is **not** to overwrite the official submission immediately. The goal is to decide whether a safer midpoint between ExtraTrees and the MLP-distance hybrid is better than choosing one extreme.


## 1. Imports and paths

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FORWARD_DIR = PROJECT_ROOT / 'data' / 'raw' / 'forward_prediction'
INVERSE_DIR = PROJECT_ROOT / 'data' / 'raw' / 'inverse_design'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
SUBMISSIONS_DIR = OUTPUTS_DIR / 'submissions'
REPORTS_DIR = PROJECT_ROOT / 'reports'

for p in [SUBMISSIONS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Forward dir:', FORWARD_DIR)
print('Submissions dir:', SUBMISSIONS_DIR)
print('Reports dir:', REPORTS_DIR)

Project root: /home/alouiyaz/projects/boom-challenge-ejecta-prediction
Forward dir: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/data/raw/forward_prediction
Submissions dir: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions
Reports dir: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports


## 2. Load train/test inputs and candidate submissions

In [2]:
input_cols = [
    'energy', 'angle_rad', 'coupling', 'strength',
    'porosity', 'gravity', 'atmosphere', 'shape_factor',
]

target_cols = [
    'P80', 'fines_frac', 'oversize_frac',
    'R95', 'R50_fines', 'R50_oversize',
]

fragmentation_cols = ['P80', 'fines_frac', 'oversize_frac']
distance_cols = ['R95', 'R50_fines', 'R50_oversize']

raw_train = pd.read_csv(FORWARD_DIR / 'train.csv')[input_cols]
raw_test = pd.read_csv(FORWARD_DIR / 'test.csv')[input_cols]
y = pd.read_csv(FORWARD_DIR / 'train_labels.csv')[target_cols]

base_path = SUBMISSIONS_DIR / 'prediction_submission.csv'
hybrid_path = SUBMISSIONS_DIR / 'prediction_submission_hybrid_mlp_distance_w_0.65.csv'

if not base_path.exists():
    raise FileNotFoundError(f'Missing base submission: {base_path}')
if not hybrid_path.exists():
    raise FileNotFoundError(f'Missing hybrid submission: {hybrid_path}')

base = pd.read_csv(base_path)
hybrid = pd.read_csv(hybrid_path)

print('Raw train:', raw_train.shape)
print('Raw test:', raw_test.shape)
print('Base submission:', base.shape, base_path)
print('Hybrid submission:', hybrid.shape, hybrid_path)

display(raw_train.head())
display(raw_test.head())
display(base.head())
display(hybrid.head())

Raw train: (2930, 8)
Raw test: (492, 8)
Base submission: (492, 7) /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission.csv
Hybrid submission: (492, 7) /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_hybrid_mlp_distance_w_0.65.csv


,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor
0,3.826405,0.818303,0.861258,1.305809,0.337215,3.71,0.781263,0.784028
1,2.828754,1.193036,0.561245,3.494501,0.058029,1.62,0.136205,0.922737
2,3.068907,0.605872,0.948860,1.366386,0.315632,3.71,0.774704,0.954922
3,2.700574,1.073708,0.713705,3.599419,0.033062,1.62,0.144204,0.932911
4,3.484022,0.863568,1.237205,1.996742,0.278207,9.81,0.414620,1.260855


,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor
0,4.387076,0.937830,1.305063,1.864599,0.162194,1.38,0.100630,0.870897
1,4.259426,0.848536,0.720038,1.803294,0.265631,4.91,0.708493,1.188009
2,4.384792,1.020636,1.003443,1.172195,0.114929,1.02,0.292809,0.957242
3,4.580126,0.888316,0.501471,2.721400,0.228540,1.02,0.787507,1.365608
4,3.607383,1.086271,1.170230,2.079364,0.184185,4.91,0.237429,0.880140


,scenario_id,P80,fines_frac,oversize_frac,R95,R50_fines,R50_oversize
0,0,120.852192,0.120046,0.208373,726.196054,704.966598,312.487130
1,1,113.668032,0.048219,0.158274,204.758075,220.793490,102.212871
2,2,149.740914,0.121343,0.384067,809.085123,808.873827,348.096898
3,3,162.227452,0.008732,0.543184,531.683980,592.186388,277.513349
4,4,137.978607,0.047156,0.357337,197.063393,243.395743,100.912752


,scenario_id,P80,fines_frac,oversize_frac,R95,R50_fines,R50_oversize
0,0,120.852192,0.120046,0.208373,1064.238127,1012.805657,448.529033
1,1,113.668032,0.048219,0.158274,180.340190,191.185816,88.486206
2,2,149.740914,0.121343,0.384067,1306.856464,1316.118523,541.277606
3,3,162.227452,0.008732,0.543184,731.377793,891.581710,418.480874
4,4,137.978607,0.047156,0.357337,152.512030,142.451662,60.233150


## 3. Submission format validation

Before blending, validate that both candidate files have the correct challenge format.

In [3]:
expected_cols = ['scenario_id'] + target_cols

def validate_submission(df, name):
    assert df.shape == (len(raw_test), 7), f'{name}: wrong shape {df.shape}'
    assert df.columns.tolist() == expected_cols, f'{name}: wrong columns'
    assert df['scenario_id'].tolist() == list(range(len(raw_test))), f'{name}: scenario_id must be 0-based index'
    assert np.isfinite(df.drop(columns=['scenario_id']).values).all(), f'{name}: contains non-finite values'
    assert (df[['P80', 'R95', 'R50_fines', 'R50_oversize']] >= 0).all().all(), f'{name}: negative size/distance'
    assert ((df[['fines_frac', 'oversize_frac']] >= 0) & (df[['fines_frac', 'oversize_frac']] <= 1)).all().all(), f'{name}: fraction outside [0, 1]'
    print(f'{name}: format OK')

validate_submission(base, 'base')
validate_submission(hybrid, 'hybrid')

base: format OK
hybrid: format OK


## 4. OOD detection with nearest-neighbor distance

We compute the distance of each test scenario to the training set in normalized input space.

A larger value means the test scenario is farther from the training distribution and therefore more exposed to extrapolation risk.


In [4]:
def compute_ood_distances(X_train_raw, X_query_raw, input_cols, n_neighbors=5):
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(X_train_raw[input_cols])
    query_scaled = scaler.transform(X_query_raw[input_cols])

    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(train_scaled)
    distances, indices = nn.kneighbors(query_scaled)

    return pd.DataFrame({
        'nearest_train_distance': distances[:, 0],
        'mean_5nn_distance': distances.mean(axis=1),
    })

test_ood = compute_ood_distances(raw_train, raw_test, input_cols, n_neighbors=5)
train_ood = compute_ood_distances(raw_train, raw_train, input_cols, n_neighbors=6)

# For train-to-train, the nearest neighbor is itself with distance 0.
# Use mean_5nn_distance as a robust baseline; it includes self if using query=train,
# but still provides a useful internal scale. We also compute a conservative threshold from test distribution.
ood_threshold_train95 = np.percentile(train_ood['mean_5nn_distance'], 95)
ood_threshold_test75 = np.percentile(test_ood['mean_5nn_distance'], 75)

print('Train 95% OOD threshold:', round(ood_threshold_train95, 4))
print('Test 75% OOD threshold:', round(ood_threshold_test75, 4))
print('Test OOD using train95:', int((test_ood['mean_5nn_distance'] > ood_threshold_train95).sum()), '/', len(test_ood))
print('Test far quartile using test75:', int((test_ood['mean_5nn_distance'] > ood_threshold_test75).sum()), '/', len(test_ood))

test_ood = test_ood.reset_index().rename(columns={'index': 'scenario_id'})
test_ood['ood_bin'] = pd.qcut(test_ood['mean_5nn_distance'], q=4, labels=['Q1_close', 'Q2', 'Q3', 'Q4_far'])
test_ood.to_csv(REPORTS_DIR / 'ood_distance_test_diagnostics.csv', index=False)

display(test_ood.describe().T)
display(test_ood.sort_values('mean_5nn_distance', ascending=False).head(20))

Train 95% OOD threshold: 0.2643
Test 75% OOD threshold: 0.7268
Test OOD using train95: 490 / 492
Test far quartile using test75: 123 / 492


,count,mean,std,min,25%,50%,75%,max
scenario_id,492.0,245.500000,142.172431,0.000000,122.750000,245.500000,368.250000,491.000000
nearest_train_distance,492.0,0.560674,0.149600,0.182295,0.435030,0.580499,0.678606,0.889880
mean_5nn_distance,492.0,0.617684,0.143318,0.234674,0.506642,0.628117,0.726834,0.947235


,scenario_id,nearest_train_distance,mean_5nn_distance,ood_bin
310,310,0.833301,0.947235,Q4_far
478,478,0.885638,0.918664,Q4_far
51,51,0.889880,0.911174,Q4_far
168,168,0.816546,0.903057,Q4_far
108,108,0.836186,0.902173,Q4_far
447,447,0.859306,0.896924,Q4_far
304,304,0.828850,0.895867,Q4_far
445,445,0.815027,0.891793,Q4_far
273,273,0.786785,0.891013,Q4_far
234,234,0.864652,0.888289,Q4_far


## 5. Compare base vs hybrid prediction shifts

The hybrid model improved CV on distance targets, but it extrapolated more strongly on the test set. Here we quantify how much it moves the distance predictions relative to the conservative ExtraTrees final model.


In [5]:
shift = test_ood.copy()
for col in target_cols:
    shift[f'{col}_base'] = base[col].values
    shift[f'{col}_hybrid'] = hybrid[col].values
    shift[f'{col}_diff'] = hybrid[col].values - base[col].values
    shift[f'{col}_abs_diff'] = np.abs(shift[f'{col}_diff'])

print('Correlation between OOD distance and absolute prediction shift:')
for col in distance_cols:
    corr = shift['mean_5nn_distance'].corr(shift[f'{col}_abs_diff'])
    print(f'{col}: {corr:.4f}')

summary_rows = []
for col in distance_cols:
    diff = shift[f'{col}_diff']
    summary_rows.append({
        'target': col,
        'base_mean': base[col].mean(),
        'hybrid_mean': hybrid[col].mean(),
        'mean_diff': diff.mean(),
        'median_abs_diff': diff.abs().median(),
        'p90_abs_diff': diff.abs().quantile(0.90),
        'p95_abs_diff': diff.abs().quantile(0.95),
        'max_abs_diff': diff.abs().max(),
        'n_abs_diff_gt_100': int((diff.abs() > 100).sum()),
        'n_abs_diff_gt_250': int((diff.abs() > 250).sum()),
    })

shift_summary = pd.DataFrame(summary_rows)
display(shift_summary)

cols = ['scenario_id', 'mean_5nn_distance', 'ood_bin']
for col in distance_cols:
    cols += [f'{col}_base', f'{col}_hybrid', f'{col}_diff', f'{col}_abs_diff']

display(shift.sort_values('R95_abs_diff', ascending=False)[cols].head(25))

Correlation between OOD distance and absolute prediction shift:
R95: 0.1651
R50_fines: 0.1613
R50_oversize: 0.1119


,target,base_mean,hybrid_mean,mean_diff,median_abs_diff,p90_abs_diff,p95_abs_diff,max_abs_diff,n_abs_diff_gt_100,n_abs_diff_gt_250
0,R95,298.530443,367.205990,68.675547,35.872930,332.709830,451.172075,810.228242,116,63
1,R50_fines,330.556280,393.496650,62.940369,57.340424,345.972356,507.756459,871.151783,132,96
2,R50_oversize,146.295647,178.261386,31.965738,23.330167,160.194100,191.370530,347.170558,104,6


,scenario_id,mean_5nn_distance,ood_bin,R95_base,R95_hybrid,R95_diff,R95_abs_diff,R50_fines_base,R50_fines_hybrid,R50_fines_diff,R50_fines_abs_diff,R50_oversize_base,R50_oversize_hybrid,R50_oversize_diff,R50_oversize_abs_diff
479,479,0.849991,Q4_far,772.512211,1582.740453,810.228242,810.228242,726.864392,1598.016175,871.151783,871.151783,330.522469,677.693027,347.170558,347.170558
250,250,0.661713,Q3,928.510613,1654.971472,726.460860,726.460860,832.051264,1560.287127,728.235864,728.235864,401.941819,665.885509,263.943690,263.943690
404,404,0.825827,Q4_far,659.167320,1346.470250,687.302930,687.302930,691.440059,1393.488106,702.048047,702.048047,293.656241,579.982499,286.326257,286.326257
185,185,0.820441,Q4_far,621.450940,1268.344065,646.893125,646.893125,674.079218,1329.993948,655.914730,655.914730,276.527204,545.439803,268.912599,268.912599
318,318,0.728975,Q4_far,843.278073,1458.464597,615.186524,615.186524,752.642424,1213.298448,460.656024,460.656024,356.904899,543.319538,186.414639,186.414639
45,45,0.690118,Q3,799.704741,1389.471151,589.766409,589.766409,728.858959,1384.757121,655.898162,655.898162,335.080871,591.396736,256.315865,256.315865
349,349,0.715488,Q3,863.535296,1448.974395,585.439099,585.439099,793.167985,1351.168717,558.000733,558.000733,352.698989,556.360146,203.661157,203.661157
416,416,0.805646,Q4_far,791.038525,1363.597193,572.558667,572.558667,745.131087,1326.259158,581.128071,581.128071,333.704629,543.986339,210.281710,210.281710
206,206,0.805969,Q4_far,665.586738,1235.478708,569.891971,569.891971,670.724899,1313.953165,643.228266,643.228266,291.218519,563.238543,272.020024,272.020024
433,433,0.796502,Q4_far,636.843751,1200.339968,563.496217,563.496217,689.823890,1300.121477,610.297586,610.297586,285.368497,530.356159,244.987663,244.987663


## 6. Shift by OOD quartile

This section checks whether the hybrid model mainly changes predictions for the farthest OOD cases. If yes, the hybrid is reacting to OOD. If no, it may be unstable even inside familiar regions.


In [6]:
quartile_summary = []
for col in distance_cols:
    grouped = shift.groupby('ood_bin').agg(
        n=('scenario_id', 'size'),
        mean_ood=('mean_5nn_distance', 'mean'),
        base_mean=(f'{col}_base', 'mean'),
        hybrid_mean=(f'{col}_hybrid', 'mean'),
        mean_diff=(f'{col}_diff', 'mean'),
        mean_abs_diff=(f'{col}_abs_diff', 'mean'),
        max_abs_diff=(f'{col}_abs_diff', 'max'),
    ).reset_index()
    grouped['target'] = col
    quartile_summary.append(grouped)

quartile_summary = pd.concat(quartile_summary, ignore_index=True)
display(quartile_summary[['target', 'ood_bin', 'n', 'mean_ood', 'base_mean', 'hybrid_mean', 'mean_diff', 'mean_abs_diff', 'max_abs_diff']])

,target,ood_bin,n,mean_ood,base_mean,hybrid_mean,mean_diff,mean_abs_diff,max_abs_diff
0,R95,Q1_close,123,0.428039,313.032453,361.108014,48.075562,69.925521,433.969553
1,R95,Q2,123,0.567561,308.171464,365.670681,57.499217,89.716358,488.442067
2,R95,Q3,123,0.681224,299.220215,369.759895,70.539679,110.021791,726.460860
3,R95,Q4_far,123,0.793915,273.697639,372.285368,98.587729,129.937007,810.228242
4,R50_fines,Q1_close,123,0.428039,348.194732,412.305827,64.111095,93.608933,530.601511
5,R50_fines,Q2,123,0.567561,341.312161,402.555356,61.243195,115.414085,542.127128
6,R50_fines,Q3,123,0.681224,325.716762,383.304048,57.587286,136.090314,728.235864
7,R50_fines,Q4_far,123,0.793915,307.001466,375.821367,68.819902,154.797221,871.151783
8,R50_oversize,Q1_close,123,0.428039,159.608852,195.081247,35.472395,45.615009,196.368379
9,R50_oversize,Q2,123,0.567561,152.614310,184.562572,31.948263,51.345904,220.776716


## 7. Create controlled blend candidates

We create several candidates between the conservative base and the aggressive hybrid.

Only distance targets are blended. Fragmentation targets remain exactly from the final ExtraTrees model.

Formula:

```text
controlled_distance = (1 - alpha) * base_distance + alpha * hybrid_distance
```

Where:

- `alpha = 0.00` → pure ExtraTrees final.
- `alpha = 1.00` → pure hybrid MLP-distance model.
- `alpha = 0.50` → midpoint, equivalent to around 67.5% ExtraTrees + 32.5% MLP internally.


In [7]:
def create_controlled_blend(base_df, hybrid_df, alpha, distance_cols, fragmentation_cols):
    blend = base_df.copy()
    for col in fragmentation_cols:
        blend[col] = base_df[col].values
    for col in distance_cols:
        blend[col] = (1 - alpha) * base_df[col].values + alpha * hybrid_df[col].values
        blend[col] = np.clip(blend[col], 0, None)
    return blend

alphas = [0.25, 0.40, 0.50, 0.60, 0.75]
blend_files = {}
blend_summaries = []

for alpha in alphas:
    candidate = create_controlled_blend(base, hybrid, alpha, distance_cols, fragmentation_cols)
    validate_submission(candidate, f'controlled_blend_alpha_{alpha:.2f}')

    out_path = SUBMISSIONS_DIR / f'prediction_submission_controlled_blend_alpha_{alpha:.2f}.csv'
    candidate.to_csv(out_path, index=False)
    blend_files[alpha] = out_path

    row = {'alpha': alpha, 'path': str(out_path)}
    for col in distance_cols:
        row[f'{col}_mean'] = candidate[col].mean()
        row[f'{col}_median'] = candidate[col].median()
        row[f'{col}_p75'] = candidate[col].quantile(0.75)
        row[f'{col}_p95'] = candidate[col].quantile(0.95)
        row[f'{col}_max'] = candidate[col].max()
    blend_summaries.append(row)

blend_summary_df = pd.DataFrame(blend_summaries)
display(blend_summary_df)
print('Saved blend candidates:')
for alpha, path in blend_files.items():
    print(alpha, path)

controlled_blend_alpha_0.25: format OK
controlled_blend_alpha_0.40: format OK
controlled_blend_alpha_0.50: format OK
controlled_blend_alpha_0.60: format OK
controlled_blend_alpha_0.75: format OK


,alpha,path,R95_mean,R95_median,R95_p75,R95_p95,R95_max,R50_fines_mean,R50_fines_median,R50_fines_p75,R50_fines_p95,R50_fines_max,R50_oversize_mean,R50_oversize_median,R50_oversize_p75,R50_oversize_p95,R50_oversize_max
0,0.25,/home/alouiyaz/projects/boom-challenge-ejecta-...,315.699329,168.481906,525.714033,902.956827,1110.125827,346.291373,200.025927,604.380533,867.336532,1014.110230,154.287082,87.607904,281.195952,382.894259,467.927742
1,0.40,/home/alouiyaz/projects/boom-challenge-ejecta-...,326.000661,163.317604,534.597301,971.246236,1219.094956,355.732428,193.092528,619.928522,935.068908,1123.345609,159.081943,84.674456,291.157898,410.354963,507.519295
2,0.50,/home/alouiyaz/projects/boom-challenge-ejecta-...,332.868216,159.304984,543.370131,1020.374924,1291.741042,362.026465,187.567113,629.519979,987.186304,1196.169196,162.278517,82.701016,297.701791,428.443965,533.913664
3,0.60,/home/alouiyaz/projects/boom-challenge-ejecta-...,339.735771,156.283265,553.933893,1058.307792,1364.387128,368.320502,181.989295,648.782200,1040.551927,1268.992782,165.475090,80.895347,303.416589,447.827905,560.308033
4,0.75,/home/alouiyaz/projects/boom-challenge-ejecta-...,350.037103,151.958390,567.002821,1114.887662,1473.356257,377.761557,172.665834,661.556107,1117.427642,1380.228229,170.269951,77.755055,316.797186,474.755744,599.899587


Saved blend candidates:
0.25 /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.25.csv
0.4 /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.40.csv
0.5 /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.50.csv
0.6 /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.60.csv
0.75 /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.75.csv


## 8. Distribution comparison: base vs hybrid vs controlled blends

In [8]:
models = {'base_extratrees': base, 'hybrid_w_0.65': hybrid}
for alpha, path in blend_files.items():
    models[f'controlled_alpha_{alpha:.2f}'] = pd.read_csv(path)

rows = []
for name, df in models.items():
    for col in distance_cols:
        rows.append({
            'model': name,
            'target': col,
            'mean': df[col].mean(),
            'std': df[col].std(),
            'min': df[col].min(),
            'median': df[col].median(),
            'p75': df[col].quantile(0.75),
            'p90': df[col].quantile(0.90),
            'p95': df[col].quantile(0.95),
            'max': df[col].max(),
        })

distribution_comparison = pd.DataFrame(rows)
display(distribution_comparison)

distribution_comparison.to_csv(REPORTS_DIR / 'controlled_blend_distribution_comparison.csv', index=False)

,model,target,mean,std,min,median,p75,p90,p95,max
0,base_extratrees,R95,298.530443,241.158254,48.776910,177.453510,508.553148,648.373860,795.225069,996.810828
1,base_extratrees,R50_fines,330.556280,235.993585,64.210862,213.098175,580.026235,670.231723,738.834486,873.066616
2,base_extratrees,R50_oversize,146.295647,108.429543,28.072745,91.697836,263.430771,291.156824,332.525738,414.330461
3,hybrid_w_0.65,R95,367.205990,387.867101,30.070443,144.903024,589.193077,1038.646374,1235.550442,1654.971472
4,hybrid_w_0.65,R50_fines,393.496650,403.381768,38.428605,156.017613,681.769264,1002.674852,1240.745131,1598.016175
5,hybrid_w_0.65,R50_oversize,178.261386,178.035458,13.158020,71.683292,328.076953,448.575919,517.474989,677.693027
6,controlled_alpha_0.25,R95,315.699329,276.157022,46.649578,168.481906,525.714033,733.610742,902.956827,1110.125827
7,controlled_alpha_0.25,R50_fines,346.291373,275.564557,62.038436,200.025927,604.380533,753.443940,867.336532,1014.110230
8,controlled_alpha_0.25,R50_oversize,154.287082,125.016964,26.713766,87.607904,281.195952,332.956577,382.894259,467.927742
9,controlled_alpha_0.40,R95,326.000661,297.850464,44.676256,163.317604,534.597301,803.535054,971.246236,1219.094956


## 9. OOD-adaptive blend candidate

This version uses a **smaller hybrid weight for close test points** and a **larger hybrid weight for far OOD points**.

This is experimental. It tries to use MLP extrapolation only where extrapolation is most likely needed.


In [9]:
def create_ood_adaptive_blend(base_df, hybrid_df, ood_df, distance_cols, fragmentation_cols, min_alpha=0.20, max_alpha=0.60):
    blend = base_df.copy()
    for col in fragmentation_cols:
        blend[col] = base_df[col].values

    scores = ood_df['mean_5nn_distance'].values
    lo = np.percentile(scores, 25)
    hi = np.percentile(scores, 90)
    scaled = (scores - lo) / (hi - lo + 1e-12)
    scaled = np.clip(scaled, 0, 1)
    alpha = min_alpha + (max_alpha - min_alpha) * scaled

    for col in distance_cols:
        blend[col] = (1 - alpha) * base_df[col].values + alpha * hybrid_df[col].values
        blend[col] = np.clip(blend[col], 0, None)

    blend['adaptive_alpha_debug'] = alpha
    return blend

adaptive = create_ood_adaptive_blend(
    base, hybrid, test_ood,
    distance_cols=distance_cols,
    fragmentation_cols=fragmentation_cols,
    min_alpha=0.20,
    max_alpha=0.60,
)

adaptive_alpha = adaptive[['scenario_id', 'adaptive_alpha_debug']].copy()
adaptive_submission = adaptive.drop(columns=['adaptive_alpha_debug'])
validate_submission(adaptive_submission, 'ood_adaptive_blend')

adaptive_path = SUBMISSIONS_DIR / 'prediction_submission_ood_adaptive_blend_alpha_0.20_0.60.csv'
adaptive_submission.to_csv(adaptive_path, index=False)
adaptive_alpha.to_csv(REPORTS_DIR / 'ood_adaptive_blend_alpha_debug.csv', index=False)

print('Saved:', adaptive_path)
display(adaptive_alpha.describe().T)
display(adaptive_submission.head())
display(adaptive_submission[distance_cols].describe().T)

ood_adaptive_blend: format OK
Saved: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_ood_adaptive_blend_alpha_0.20_0.60.csv


,count,mean,std,min,25%,50%,75%,max
scenario_id,492.0,245.500000,142.172431,0.0,122.750000,245.500000,368.250000,491.0
adaptive_alpha_debug,492.0,0.374075,0.147049,0.2,0.200375,0.368838,0.506043,0.6


,scenario_id,P80,fines_frac,oversize_frac,R95,R50_fines,R50_oversize
0,0,120.852192,0.120046,0.208373,879.903826,844.941056,374.345400
1,1,113.668032,0.048219,0.158274,199.874498,214.871955,99.467538
2,2,149.740914,0.121343,0.384067,1045.074616,1049.354563,439.682358
3,3,162.227452,0.008732,0.543184,571.622743,652.065452,305.706854
4,4,137.978607,0.047156,0.357337,179.767194,204.206170,85.119689


,count,mean,std,min,25%,50%,75%,max
R95,492.0,327.133550,302.132929,40.464124,97.864575,164.663765,529.450783,1258.649156
R50_fines,492.0,354.400257,303.531438,58.184816,123.299538,195.370609,605.401438,1249.555462
R50_oversize,492.0,158.137348,135.776475,21.308437,51.190738,85.665521,282.864169,538.824804


## 10. Final candidate comparison table

Use this table to decide which candidate has a reasonable balance between stability and OOD extrapolation.


In [10]:
final_models = dict(models)
final_models['ood_adaptive_alpha_0.20_0.60'] = adaptive_submission

rows = []
for name, df in final_models.items():
    row = {'model': name}
    for col in distance_cols:
        row[f'{col}_mean'] = df[col].mean()
        row[f'{col}_median'] = df[col].median()
        row[f'{col}_p75'] = df[col].quantile(0.75)
        row[f'{col}_p95'] = df[col].quantile(0.95)
        row[f'{col}_max'] = df[col].max()
    rows.append(row)

candidate_comparison = pd.DataFrame(rows)
display(candidate_comparison)
candidate_comparison.to_csv(REPORTS_DIR / 'final_forward_candidate_distribution_comparison.csv', index=False)

print('Candidate files created:')
print('Base official:', base_path)
print('Hybrid aggressive:', hybrid_path)
for alpha, path in blend_files.items():
    print(f'Controlled alpha {alpha:.2f}:', path)
print('OOD adaptive:', adaptive_path)

,model,R95_mean,R95_median,R95_p75,R95_p95,R95_max,R50_fines_mean,R50_fines_median,R50_fines_p75,R50_fines_p95,R50_fines_max,R50_oversize_mean,R50_oversize_median,R50_oversize_p75,R50_oversize_p95,R50_oversize_max
0,base_extratrees,298.530443,177.453510,508.553148,795.225069,996.810828,330.556280,213.098175,580.026235,738.834486,873.066616,146.295647,91.697836,263.430771,332.525738,414.330461
1,hybrid_w_0.65,367.205990,144.903024,589.193077,1235.550442,1654.971472,393.496650,156.017613,681.769264,1240.745131,1598.016175,178.261386,71.683292,328.076953,517.474989,677.693027
2,controlled_alpha_0.25,315.699329,168.481906,525.714033,902.956827,1110.125827,346.291373,200.025927,604.380533,867.336532,1014.110230,154.287082,87.607904,281.195952,382.894259,467.927742
3,controlled_alpha_0.40,326.000661,163.317604,534.597301,971.246236,1219.094956,355.732428,193.092528,619.928522,935.068908,1123.345609,159.081943,84.674456,291.157898,410.354963,507.519295
4,controlled_alpha_0.50,332.868216,159.304984,543.370131,1020.374924,1291.741042,362.026465,187.567113,629.519979,987.186304,1196.169196,162.278517,82.701016,297.701791,428.443965,533.913664
5,controlled_alpha_0.60,339.735771,156.283265,553.933893,1058.307792,1364.387128,368.320502,181.989295,648.782200,1040.551927,1268.992782,165.475090,80.895347,303.416589,447.827905,560.308033
6,controlled_alpha_0.75,350.037103,151.958390,567.002821,1114.887662,1473.356257,377.761557,172.665834,661.556107,1117.427642,1380.228229,170.269951,77.755055,316.797186,474.755744,599.899587
7,ood_adaptive_alpha_0.20_0.60,327.133550,164.663765,529.450783,976.641967,1258.649156,354.400257,195.370609,605.401438,932.897330,1249.555462,158.137348,85.665521,282.864169,417.213472,538.824804


Candidate files created:
Base official: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission.csv
Hybrid aggressive: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_hybrid_mlp_distance_w_0.65.csv
Controlled alpha 0.25: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.25.csv
Controlled alpha 0.40: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.40.csv
Controlled alpha 0.50: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.50.csv
Controlled alpha 0.60: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.60.csv
Controlled alpha 0.75: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/subm

## 11. Decision notes

Recommended interpretation:

- If the hybrid model creates very large distance values for many test scenarios, it is aggressive and risky.
- If the base ExtraTrees model appears too conservative for far-OOD test points, a controlled blend can be a safer compromise.
- The most useful candidate is usually the one that increases distance predictions moderately without creating extreme maximum values.

Do **not** overwrite `outputs/submissions/prediction_submission.csv` inside this notebook.

After reviewing the distribution comparison, if we decide to submit one candidate, copy it manually to the official filename.

Example:

```bash
cp outputs/submissions/prediction_submission_controlled_blend_alpha_0.50.csv outputs/submissions/prediction_submission.csv
```

or

```bash
cp outputs/submissions/prediction_submission_ood_adaptive_blend_alpha_0.20_0.60.csv outputs/submissions/prediction_submission.csv
```
